In [77]:
import pandas as pd
import numpy as np
from ta.trend import SMAIndicator

start_date = '2026-5-1'
end_date = '2026-9-15'

df_d1 = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_sma = SMAIndicator(df_d1['Close'], 20)
df_d1['SMA'] = indicator_sma.sma_indicator();

previous_row = None
previous_row_list = []

def add_new_row(row):
    if len(previous_row_list) == 10:
        previous_row_list.pop(0)
        previous_row_list.append(row)
    else:
        previous_row_list.append(row)

    return

def get_last():
    return previous_row_list[9]

def get_gap_sma():
    return previous_row_list[9]['SMA'] - previous_row_list[0]['SMA']

for index, row in df_d1.iterrows():
    if not np.isnan(row['SMA']):

        if len(previous_row_list) < 10:
            add_new_row(row)
            continue

        if len(previous_row_list) == 10:
            previous_row = get_last()
            if previous_row.Close < row['SMA'] and row.Close > row['SMA']:
                print(f"date = {index} & crossed SMA bullish close = {row.Close} & sma = {row['SMA']} & gap = {get_gap_sma()}")

            if previous_row.Close > row['SMA'] and row.Close < row['SMA']:
                print(f"date = {index} & crossed SMA bearish close = {row.Close} & sma = {row['SMA']} & gap = {get_gap_sma()}")

            print(f"date = {index} close = {row.Close} & sma = {row['SMA']} & gap = {get_gap_sma()}")
            

        add_new_row(row)
        # print(row)
        # print(len(previous_row_list))



date = 2026-06-04 00:00:00 close = 214.749 & sma = 213.75809999999996 & gap = 0.5751499999999794
date = 2026-06-05 00:00:00 close = 213.84 & sma = 213.77399999999997 & gap = 0.5751499999999794
date = 2026-06-07 00:00:00 close = 213.804 & sma = 213.86025 & gap = 0.5320999999999856
date = 2026-06-08 00:00:00 close = 213.573 & sma = 213.9641 & gap = 0.5509000000000128
date = 2026-06-09 00:00:00 & crossed SMA bullish close = 214.458 & sma = 214.11624999999998 & gap = 0.6450000000000102
date = 2026-06-09 00:00:00 close = 214.458 & sma = 214.11624999999998 & gap = 0.6450000000000102
date = 2026-06-10 00:00:00 close = 214.49 & sma = 214.1718 & gap = 0.7207499999999811
date = 2026-06-11 00:00:00 close = 214.8 & sma = 214.25969999999998 & gap = 0.6890500000000088
date = 2026-06-12 00:00:00 close = 214.763 & sma = 214.32565 & gap = 0.7098999999999762
date = 2026-06-14 00:00:00 close = 215.135 & sma = 214.40325 & gap = 0.6771000000000242
date = 2026-06-15 00:00:00 close = 214.97 & sma = 214.46314

In [2]:
from backtesting import Strategy

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    # range_to_consider = 5
    # tp_multiple = 2.5
    # stop_trade_hour = 18
    # close_range_hour = 8

    # variables
    range_to_consider = 3
    tp_multiple = 5.5
    stop_trade_hour = 18
    close_range_hour = 8

    # fixed variables
    my_equity = 10000
    max_leverage = 4
    risk_percent = 0.01
    open_range_hour = close_range_hour - range_to_consider

    def init(self):
            self.current_day = None
            self.opening_range_high = None
            self.opening_range_low = None
            if global_range_to_consider != 0:
                self.range_to_consider = global_range_to_consider
    
            if global_tp_multiple != 0:
                self.tp_multiple = global_tp_multiple
    
            if global_stop_trade_hour != 0:
                self.stop_trade_hour = global_stop_trade_hour
    
            if global_close_range_hour != 0:
                self.close_range_hour = global_close_range_hour
    
            if global_close_range_hour != 0 and global_range_to_consider != 0:
                self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.trade_taken_date = None
        self.opening_range_high = None
        self.opening_range_low = None
        # print("---------------------------------------------------")

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        # print(self.data['dl_direction'][-1])
        # print(self.data['ATR'][-1])

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['dl_direction'][-1] == 1
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['dl_direction'][-1] == 2

            # c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high
            # c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                if c_long and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
            # else:
            #     # trade is taken and done
            #     if self.trade_taken_date == current_bar_date and self.trades_count == len(self.trades):
            #         self.current_day_position_taked = False

        

In [3]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

timezone = "Europe/London"

start_date = '2020-1-1'
end_date = '2026-9-16'

df_d1 = pd.read_csv("./Trading Data/GBPJPY_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_ema_5 = EMAIndicator(df_d1['Close'], 5)
indicator_ema_8 = EMAIndicator(df_d1['Close'], 8)
df_d1['EMA5'] = indicator_ema_5.ema_indicator();
df_d1['EMA8'] = indicator_ema_8.ema_indicator();

df_h1 = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

previous_row = None
previous_direction = 0
d1_tf_direction_list = []

for index, row in df_d1.iterrows():
    df_by_day = df_h1.loc[f'{index.date()}' : f'{index.date()}']

    if index.weekday() == 5 or index.weekday() == 6:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(0)
        continue
    else:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(previous_direction)
    
    if not np.isnan(row['EMA5']) and not np.isnan(row['EMA8']):
        if previous_row is not None:
            if previous_row['EMA5'] < previous_row['EMA8'] and row['EMA5'] > row['EMA8']:
                # print(f"{index} crossed BULLISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 1

            if previous_row['EMA5'] > previous_row['EMA8'] and row['EMA5'] < row['EMA8']:
                # print(f"{index} crossed BEARISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 2

        previous_row = row

df_h1['dl_direction'] = d1_tf_direction_list
indicator_atr = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr._atr
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

Start                     2020-01-01 22:00...
End                       2026-09-17 00:00...
Duration                   2450 days 01:00:00
Exposure Time [%]                    67.57235
Equity Final [$]                    18662.697
Equity Peak [$]                     24374.444
Return [%]                           86.62697
Buy & Hold Return [%]                45.04616
Return (Ann.) [%]                     7.78242
Volatility (Ann.) [%]                22.34645
CAGR [%]                                9.748
Sharpe Ratio                          0.34826
Sortino Ratio                         0.61936
Calmar Ratio                          0.23672
Alpha [%]                            78.43229
Beta                                  0.18192
Max. Drawdown [%]                   -32.87568
Avg. Drawdown [%]                    -2.87131
Max. Drawdown Duration      628 days 04:00:00
Avg. Drawdown Duration       19 days 22:00:00
# Trades                                  749
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1261', ...)

In [170]:
import time

start_date = '2020-1-1'
end_date = '2026-9-16'

range_to_consider_list = [5]
close_range_hour_list = [6,7,8,9]
stop_trade_hour_list = [18,19,20,21]
tp_multiple_list = [1,1.5,2,2.5,3,3.5,4,4.5,5,5.5,6]

with open(f"./results/result_{time.time()}.txt", "a") as f:
    for range in range_to_consider_list:
        for close_range_hour in close_range_hour_list:
            for stop_trade_hour in stop_trade_hour_list:
                for tp_multiple in tp_multiple_list:
                    # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                    global_range_to_consider = range
                    global_close_range_hour = close_range_hour
                    global_stop_trade_hour = stop_trade_hour
                    global_tp_multiple = tp_multiple


                    bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                    stats = bt.run()

                    f.write(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                    f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                    f.write("----------------------------------------------\n")

                    print(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")


range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 1 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 1.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 2 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 2.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 3 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 3.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 4 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 4.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 5.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multipl